# Data Validation Pipeline: URL Alignment Check

**Objective**: Validate 100% alignment between:
1. Perceptually adjusted URLs (source of truth)
2. RAG indices (summaries + QA pairs)
3. BBOA cache records (with Lamia image metadata)

**Victory Condition**: All perceptually adjusted URLs have corresponding RAG indices AND BBOA records, enabling media linkage → clustering → Kaliya agenda analysis.

## 1. Set Up Environment and Paths

In [11]:
import json
import os
from pathlib import Path
from collections import defaultdict
from typing import Set, Dict, List
import pandas as pd
import re

# Define paths
PERCEPTUALLY_ADJUSTED_DIR = Path("/home/electric/workspace/RapporteurIndices/src/notebooks/working_dir")
INDICES_BASE = Path("/mnt/workspace_2/rapporteur_indices")  # Note: corrected typo from "rapportuer"
BBOA_CACHE_BASE = Path("/mnt/workspace_2/bboa_cache")

# Helper: URL normalization (strip scheme, query, fragment, trailing slash)
def normalize_url(url: str) -> str:
    """Normalize URL for comparison."""
    if not url:
        return ""
    normalized = re.sub(r"^https?://", "", url.strip().lower())
    normalized = normalized.split("#")[0].split("?")[0]
    return normalized.rstrip("/")

print("✓ Environment configured")
print(f"  Perceptually adjusted: {PERCEPTUALLY_ADJUSTED_DIR}")
print(f"  Indices base:          {INDICES_BASE}")
print(f"  BBOA cache base:       {BBOA_CACHE_BASE}")

✓ Environment configured
  Perceptually adjusted: /home/electric/workspace/RapporteurIndices/src/notebooks/working_dir
  Indices base:          /mnt/workspace_2/rapporteur_indices
  BBOA cache base:       /mnt/workspace_2/bboa_cache


In [12]:
def load_perceptually_adjusted_urls() -> Set[str]:
    """
    Load all URLs from perceptually_adjusted files.
    These are the primary URLs that MUST be indexed.
    """
    urls = set()
    
    if not PERCEPTUALLY_ADJUSTED_DIR.exists():
        print(f"❌ Directory not found: {PERCEPTUALLY_ADJUSTED_DIR}")
        return urls
    
    # Find all perceptually_adjusted JSONL files
    for file_path in PERCEPTUALLY_ADJUSTED_DIR.glob("*perceptually_adjusted*.jsonl"):
        print(f"  Loading {file_path.name}...")
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    url = rec.get("source_url") or rec.get("url") or rec.get("link")
                    if url:
                        urls.add(normalize_url(url))
                except json.JSONDecodeError:
                    continue
    
    print(f"✓ Loaded {len(urls)} unique perceptually adjusted URLs")
    return urls

perceptually_adjusted_urls = load_perceptually_adjusted_urls()

  Loading vance_legacy_media_dedup_perceptually_adjusted.jsonl...
  Loading dow_legacy_media_dedup_perceptually_adjusted.jsonl...
  Loading trump_legacy_media_dedup_perceptually_adjusted.jsonl...
  Loading dow_legacy_media_dedup_perceptually_adjusted_2026_v2.jsonl...
  Loading trump_legacy_media_dedup_perceptually_adjusted_2026_v2.jsonl...
✓ Loaded 1679 unique perceptually adjusted URLs


In [13]:
def load_indexed_urls() -> Dict[str, Set[str]]:
    """
    Load all URLs from RAG indices (summaries + QA pairs).
    Returns dict with keys: 'summaries', 'qa_pairs', 'all'
    """
    results = {
        "summaries": set(),
        "qa_pairs": set(),
        "all": set()
    }
    
    if not INDICES_BASE.exists():
        print(f"❌ Indices directory not found: {INDICES_BASE}")
        return results
    
    # Scan all index folders (e.g., x_indices_hegseth, x_indices_trump, etc.)
    for index_dir in INDICES_BASE.iterdir():
        if not index_dir.is_dir():
            continue
        
        print(f"  Scanning {index_dir.name}...")
        
        # Load summaries
        for summary_file in index_dir.glob("research_summaries*.jsonl"):
            with open(summary_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                        url = rec.get("source_url")
                        if url:
                            norm_url = normalize_url(url)
                            results["summaries"].add(norm_url)
                            results["all"].add(norm_url)
                    except json.JSONDecodeError:
                        continue
        
        # Load QA pairs
        for qa_file in index_dir.glob("research_qa_pairs*.jsonl"):
            with open(qa_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                        # QA pairs have source_url in document_metadata
                        url = rec.get("document_metadata", {}).get("source_url")
                        if url:
                            norm_url = normalize_url(url)
                            results["qa_pairs"].add(norm_url)
                            results["all"].add(norm_url)
                    except json.JSONDecodeError:
                        continue
    
    print(f"✓ Found {len(results['summaries'])} URLs with summaries")
    print(f"✓ Found {len(results['qa_pairs'])} URLs with QA pairs")
    print(f"✓ Total unique indexed URLs: {len(results['all'])}")
    return results

indexed_urls = load_indexed_urls()

  Scanning staging...
✓ Found 0 URLs with summaries
✓ Found 0 URLs with QA pairs
✓ Total unique indexed URLs: 0


In [14]:
# Find URLs missing from indices
missing_from_indices = perceptually_adjusted_urls - indexed_urls["all"]
missing_summaries = perceptually_adjusted_urls - indexed_urls["summaries"]
missing_qa_pairs = perceptually_adjusted_urls - indexed_urls["qa_pairs"]

# Calculate match rates
total = len(perceptually_adjusted_urls)
indexed_count = len(perceptually_adjusted_urls & indexed_urls["all"])
match_rate = (indexed_count / total * 100) if total > 0 else 0

print("="*60)
print("INDICES COVERAGE ANALYSIS")
print("="*60)
print(f"Total perceptually adjusted URLs:  {total}")
print(f"URLs with index files:             {indexed_count}")
print(f"Match rate:                        {match_rate:.1f}%")
print(f"\nMissing from indices:              {len(missing_from_indices)}")
print(f"  - Missing summaries:             {len(missing_summaries)}")
print(f"  - Missing QA pairs:              {len(missing_qa_pairs)}")

# Show sample of missing URLs
if missing_from_indices:
    print(f"\nSample missing URLs (first 10):")
    for url in list(missing_from_indices)[:10]:
        print(f"  - {url}")

INDICES COVERAGE ANALYSIS
Total perceptually adjusted URLs:  1679
URLs with index files:             0
Match rate:                        0.0%

Missing from indices:              1679
  - Missing summaries:             1679
  - Missing QA pairs:              1679

Sample missing URLs (first 10):
  - thehill.com/newsletters/the-gavel/5622466-trump-targets-democrats-sedition
  - www.independent.co.uk/news/world/americas/us-politics/hegseth-department-of-war-defense-trump-b2843354.html
  - www.lemonde.fr/en/united-states/article/2025/01/20/trump-inauguration-to-be-first-attended-by-foreign-leaders_6737215_133.html
  - www.timesofisrael.com/liveblog-december-22-2025
  - www.axios.com/2025/07/30/trump-pentagon-think-tanks-aspen
  - www.theguardian.com/world/2025/dec/13/israel-says-its-military-killed-hamas-commander-raed-saed-in-gaza-city-strike
  - www.jpost.com/israel-news/article-859293
  - www.politico.com/news/2025/09/17/dod-kirk-pentagon-speech-00569926
  - theintercept.com/2025/05/02

In [15]:
def load_bboa_cache_urls() -> Set[str]:
    """
    Load all URLs from BBOA cache records.
    BBOA cache should contain enriched metadata + Lamia image records.
    """
    urls = set()
    
    if not BBOA_CACHE_BASE.exists():
        print(f"❌ BBOA cache not found: {BBOA_CACHE_BASE}")
        return urls
    
    # Scan all cache subdirectories
    for cache_dir in BBOA_CACHE_BASE.rglob("*.jsonl"):
        print(f"  Loading {cache_dir.name}...")
        with open(cache_dir, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    # BBOA cache may have different field names
                    url = (rec.get("source_url") or 
                           rec.get("url") or 
                           rec.get("link") or
                           rec.get("document_url"))
                    if url:
                        urls.add(normalize_url(url))
                except json.JSONDecodeError:
                    continue
    
    print(f"✓ Found {len(urls)} URLs in BBOA cache")
    return urls

bboa_cache_urls = load_bboa_cache_urls()

# Compare against perceptually adjusted URLs
missing_from_bboa = perceptually_adjusted_urls - bboa_cache_urls
bboa_coverage = len(perceptually_adjusted_urls & bboa_cache_urls)
bboa_rate = (bboa_coverage / total * 100) if total > 0 else 0

print("\n" + "="*60)
print("BBOA CACHE COVERAGE ANALYSIS")
print("="*60)
print(f"Total perceptually adjusted URLs:  {total}")
print(f"URLs with BBOA records:            {bboa_coverage}")
print(f"BBOA coverage rate:                {bboa_rate:.1f}%")
print(f"Missing from BBOA:                 {len(missing_from_bboa)}")

if missing_from_bboa:
    print(f"\nSample missing BBOA records (first 10):")
    for url in list(missing_from_bboa)[:10]:
        print(f"  - {url}")

❌ BBOA cache not found: /mnt/workspace_2/bboa_cache

BBOA CACHE COVERAGE ANALYSIS
Total perceptually adjusted URLs:  1679
URLs with BBOA records:            0
BBOA coverage rate:                0.0%
Missing from BBOA:                 1679

Sample missing BBOA records (first 10):
  - thehill.com/newsletters/the-gavel/5622466-trump-targets-democrats-sedition
  - www.independent.co.uk/news/world/americas/us-politics/hegseth-department-of-war-defense-trump-b2843354.html
  - www.lemonde.fr/en/united-states/article/2025/01/20/trump-inauguration-to-be-first-attended-by-foreign-leaders_6737215_133.html
  - www.timesofisrael.com/liveblog-december-22-2025
  - www.axios.com/2025/07/30/trump-pentagon-think-tanks-aspen
  - www.theguardian.com/world/2025/dec/13/israel-says-its-military-killed-hamas-commander-raed-saed-in-gaza-city-strike
  - www.jpost.com/israel-news/article-859293
  - www.politico.com/news/2025/09/17/dod-kirk-pentagon-speech-00569926
  - theintercept.com/2025/05/02/trump-yemen-war-

In [16]:
# Export missing URLs to files for downstream processing
output_dir = Path("/home/electric/workspace/RapporteurApp/data_validation_outputs")
output_dir.mkdir(exist_ok=True, parents=True)

# 1. Missing index entries (need indexing via research-indexing-consumers)
missing_indices_file = output_dir / "missing_indices.jsonl"
with open(missing_indices_file, "w", encoding="utf-8") as f:
    for url in sorted(missing_from_indices):
        f.write(json.dumps({"source_url": f"https://{url}"}) + "\n")
print(f"✓ Exported {len(missing_from_indices)} URLs to {missing_indices_file}")

# 2. Missing BBOA records (need BBOA consumers + Lamia)
missing_bboa_file = output_dir / "missing_bboa.jsonl"
with open(missing_bboa_file, "w", encoding="utf-8") as f:
    for url in sorted(missing_from_bboa):
        f.write(json.dumps({"source_url": f"https://{url}"}) + "\n")
print(f"✓ Exported {len(missing_from_bboa)} URLs to {missing_bboa_file}")

# 3. Combined gap analysis report
gap_report = output_dir / "gap_analysis_report.json"
report = {
    "timestamp": pd.Timestamp.now().isoformat(),
    "total_perceptually_adjusted_urls": total,
    "indices_coverage": {
        "count": indexed_count,
        "rate_pct": round(match_rate, 2),
        "missing_count": len(missing_from_indices),
        "missing_summaries": len(missing_summaries),
        "missing_qa_pairs": len(missing_qa_pairs)
    },
    "bboa_coverage": {
        "count": bboa_coverage,
        "rate_pct": round(bboa_rate, 2),
        "missing_count": len(missing_from_bboa)
    },
    "victory_condition_met": (len(missing_from_indices) == 0 and len(missing_from_bboa) == 0)
}
with open(gap_report, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print(f"✓ Saved gap analysis report to {gap_report}")

print("\n" + "="*60)
print("Files exported to:", output_dir)
print("  - missing_indices.jsonl    → Feed to research-indexing-consumers")
print("  - missing_bboa.jsonl       → Feed to BBOA consumers + Lamia")
print("  - gap_analysis_report.json → Summary statistics")
print("="*60)

✓ Exported 1679 URLs to /home/electric/workspace/RapporteurApp/data_validation_outputs/missing_indices.jsonl
✓ Exported 1679 URLs to /home/electric/workspace/RapporteurApp/data_validation_outputs/missing_bboa.jsonl
✓ Saved gap analysis report to /home/electric/workspace/RapporteurApp/data_validation_outputs/gap_analysis_report.json

Files exported to: /home/electric/workspace/RapporteurApp/data_validation_outputs
  - missing_indices.jsonl    → Feed to research-indexing-consumers
  - missing_bboa.jsonl       → Feed to BBOA consumers + Lamia
  - gap_analysis_report.json → Summary statistics


In [17]:
# Reload data after reprocessing
print("Reloading indices and BBOA cache...")
indexed_urls_post = load_indexed_urls()
bboa_cache_urls_post = load_bboa_cache_urls()

# Recalculate gaps
missing_indices_post = perceptually_adjusted_urls - indexed_urls_post["all"]
missing_bboa_post = perceptually_adjusted_urls - bboa_cache_urls_post

# Victory condition check
indices_aligned = len(missing_indices_post) == 0
bboa_aligned = len(missing_bboa_post) == 0
victory = indices_aligned and bboa_aligned

print("\n" + "="*60)
print("POST-PROCESSING VALIDATION")
print("="*60)
print(f"Indices 100% aligned:  {'✓ YES' if indices_aligned else '✗ NO'}")
print(f"  Missing count:       {len(missing_indices_post)}")
print(f"\nBBOA 100% aligned:     {'✓ YES' if bboa_aligned else '✗ NO'}")
print(f"  Missing count:       {len(missing_bboa_post)}")
print(f"\n{'='*60}")
print(f"VICTORY CONDITION:     {'✓ MET' if victory else '✗ NOT MET'}")
print(f"{'='*60}")

if victory:
    print("\n✓✓✓ All perceptually adjusted URLs have RAG indices + BBOA records!")
    print("    Ready for: Media Linkage → Clustering → Kaliya Analysis")
else:
    print("\n✗ Gap still exists. Review missing_indices.jsonl and missing_bboa.jsonl")

Reloading indices and BBOA cache...
  Scanning staging...
✓ Found 0 URLs with summaries
✓ Found 0 URLs with QA pairs
✓ Total unique indexed URLs: 0
❌ BBOA cache not found: /mnt/workspace_2/bboa_cache

POST-PROCESSING VALIDATION
Indices 100% aligned:  ✗ NO
  Missing count:       1679

BBOA 100% aligned:     ✗ NO
  Missing count:       1679

VICTORY CONDITION:     ✗ NOT MET

✗ Gap still exists. Review missing_indices.jsonl and missing_bboa.jsonl


In [18]:
def run_kaliya_agenda_analysis():
    """
    Execute Kaliya agenda analysis on clustered media.
    Generates reports by subsection.
    """
    print("Running Kaliya agenda analysis by subsection...")
    
    # CLI command for Kaliya stage
    cmd = [
        "python", "-m", "src.components.publisher.rapporteur_cli",
        "docs/CultureTechnologyProcurementMedia/document.yaml",
        "--kaliya", "-v"
    ]
    
    print(f"Command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd="/home/electric/workspace/RapporteurApp")
    
    if result.returncode == 0:
        print("✓ Kaliya analysis completed")
        # Extract summary
        for line in result.stdout.split("\n"):
            if "subsection" in line.lower() or "agenda" in line.lower():
                print(f"  {line.strip()}")
    else:
        print(f"✗ Kaliya analysis failed with exit code {result.returncode}")
        print(result.stderr[-500:])
    
    return result.returncode == 0

# Uncomment to run after clustering
# kaliya_success = run_kaliya_agenda_analysis()

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print("Next steps:")
print("1. Review Kaliya agenda analysis outputs")
print("2. Run conclude stage for final document generation")
print("3. Run section-by-section conclusion generation if needed")
print("="*60)


PIPELINE COMPLETE
Next steps:
1. Review Kaliya agenda analysis outputs
2. Run conclude stage for final document generation
3. Run section-by-section conclusion generation if needed


## Export Grounding Package to Excel

Load the full Phase 4 grounding bundle (`phase4_culture_and_media_*.bundle.jsonl`) — 293 articles with media framing analysis, risk ratings, metadata, and cluster assignments — plus 445 grounding events. Export to a multi-sheet `.xlsx` workbook.

In [22]:
import json, re
from pathlib import Path
from collections import Counter, defaultdict
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter

# ── Paths ────────────────────────────────────────────────────────────────────
BUNDLE = Path("/home/electric/workspace/RapporteurApp/cache/"
              "phase4_culture_and_media_20260328_183618.bundle.jsonl")
TABLES_DIR = Path("/mnt/workspace_2/bboa/documents/culture_and_media/tables")
OUTPUT = output_dir / "grounding_package.xlsx"

# ── Styles ───────────────────────────────────────────────────────────────────
HDR_FONT = Font(bold=True, color="FFFFFF", size=11)
HDR_FILL = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
LINK_FONT = Font(color="0563C1", underline="single")
WRAP_TOP = Alignment(wrap_text=True, vertical="top")
RISK_FILLS = {
    "LOW":     PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"),
    "MEDIUM":  PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid"),
    "HIGH":    PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid"),
    "EXTREME": PatternFill(start_color="FF4444", end_color="FF4444", fill_type="solid"),
}
CLUSTER_FILLS = {
    "A": PatternFill(start_color="DAEEF3", end_color="DAEEF3", fill_type="solid"),
    "B": PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid"),
    "C": PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid"),
    "D": PatternFill(start_color="EDEDED", end_color="EDEDED", fill_type="solid"),
    "E": PatternFill(start_color="D9E2F3", end_color="D9E2F3", fill_type="solid"),
}

def _style_hdr(ws, n):
    for c in range(1, n+1):
        cell = ws.cell(row=1, column=c)
        cell.font, cell.fill = HDR_FONT, HDR_FILL
        cell.alignment = Alignment(wrap_text=True, vertical="center")
    ws.freeze_panes = "A2"
    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions

def _auto_w(ws, mn=10, mx=65):
    for col in ws.columns:
        L = get_column_letter(col[0].column)
        ws.column_dimensions[L].width = min(max(mn, max((min(len(str(c.value or "")), mx) for c in col), default=mn)+2), mx)

def _stance_bucket(s):
    s = s.lower()
    if "unsupportive" in s: return "Unsupportive"
    if "supportive" in s: return "Supportive"
    if "neutral" in s: return "Neutral"
    return "Unknown"

# ── Load bundle ──────────────────────────────────────────────────────────────
articles = []
with open(BUNDLE, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        raw = json.loads(line)
        mf = raw.get("media_framing") or {}
        risk = mf.get("risk") or {}
        dm = raw.get("doc_metadata") or {}
        articles.append({
            "#": i,
            "title": raw.get("title", ""),
            "url": raw.get("url", ""),
            "publisher": raw.get("publisher", ""),
            "listed_publisher": raw.get("listed_publisher", ""),
            "author": raw.get("author", ""),
            "date": raw.get("date_publication", ""),
            "eo": raw.get("eo_identifier", ""),
            "source_type": raw.get("source_type", ""),
            "reconstituted": raw.get("reconstituted", False),
            "qa_count": raw.get("qa_count", 0),
            "salience": raw.get("consumer_salience"),
            "search_rank": raw.get("mean_search_engine_rank"),
            # media framing
            "risk_rating": risk.get("rating", ""),
            "risk_reasoning": risk.get("reasoning", ""),
            "stance": mf.get("supportive_unsupportive", ""),
            "spin": mf.get("spin", ""),
            "factualness": mf.get("factualness", ""),
            "convergence": mf.get("convergence_divergence", ""),
            "linked_to_dow": mf.get("linked_to_dow", ""),
            # doc_metadata
            "dm_type": dm.get("type", ""),
            "dm_venue": dm.get("publication_venue", ""),
            "dm_doi": dm.get("doi", ""),
            "dm_isbn": dm.get("isbn", ""),
            # cluster filled below
            "cluster": "",
        })
print(f"✓ {len(articles)} articles loaded from bundle")

# ── Cluster assignments ──────────────────────────────────────────────────────
cluster_map = {}
for fp in sorted(TABLES_DIR.glob("_full_table_cluster_*.md")):
    cl = fp.stem[-1]
    for line in fp.read_text(encoding="utf-8").splitlines():
        m = re.match(r'^\|\s*(\d+)\s*\|', line.strip())
        if m:
            cluster_map[int(m.group(1))] = cl
for a in articles:
    a["cluster"] = cluster_map.get(a["#"], "")
print(f"  {sum(1 for a in articles if a['cluster'])}/{len(articles)} in clusters")

# ── Grounding events ─────────────────────────────────────────────────────────
_GH = re.compile(r'^##\s*\[(?P<tag>G\d+)\]\s*(?:EO\s*(?P<eo>\d+))?[:\s]*(?P<title>.*?)\s*$')
_GF = re.compile(r'^-\s*\*\*(?P<k>[^*]+)\*\*:?\s*(?P<v>.*)$')
ge_records = []
ge_file = TABLES_DIR / "23-appendix_grounding_events_reference.md"
if ge_file.exists():
    cur = None
    for line in ge_file.read_text(encoding="utf-8").splitlines():
        m = _GH.match(line.strip())
        if m:
            if cur: ge_records.append(cur)
            cur = {"Tag": m.group("tag"), "EO": m.group("eo") or "", "EO Title": (m.group("title") or "").strip(" —–-")}
            continue
        if cur:
            fm = _GF.match(line.strip())
            if fm: cur[fm.group("k").strip()] = fm.group("v").strip()
    if cur: ge_records.append(cur)
print(f"  {len(ge_records)} grounding events parsed")

# ── Build workbook ───────────────────────────────────────────────────────────
wb = Workbook(); wb.remove(wb.active)

# Article column spec
COLS = [
    "#", "Cluster", "EO", "Title", "Publisher", "Listed Publisher", "Author", "Date",
    "Source Type", "Reconstituted", "Risk Rating", "Stance Bucket", "Salience",
    "Search Rank", "QA Count", "Doc Type", "Venue",
    "Stance (full)", "Spin", "Factualness", "Convergence/Divergence",
    "Linked to DoW", "Risk Reasoning", "DOI", "ISBN", "URL",
]
WIDE_COLS = {"Stance (full)", "Spin", "Factualness", "Convergence/Divergence", "Linked to DoW", "Risk Reasoning"}

def _write_articles(name, recs):
    ws = wb.create_sheet(title=name[:31])
    ws.append(COLS)
    for a in recs:
        ws.append([
            a["#"], a["cluster"], a["eo"], a["title"], a["publisher"],
            a["listed_publisher"], a["author"], a["date"], a["source_type"],
            a["reconstituted"], a["risk_rating"], _stance_bucket(a["stance"]),
            round(a["salience"], 3) if a["salience"] else None,
            round(a["search_rank"], 2) if a["search_rank"] else None,
            a["qa_count"], a["dm_type"], a["dm_venue"],
            a["stance"], a["spin"], a["factualness"], a["convergence"],
            a["linked_to_dow"], a["risk_reasoning"], a["dm_doi"], a["dm_isbn"],
            a["url"],
        ])
    _style_hdr(ws, len(COLS))
    # Hyperlink titles
    ti, ui = COLS.index("Title")+1, COLS.index("URL")+1
    for r in range(2, ws.max_row+1):
        u = str(ws.cell(row=r, column=ui).value or "")
        if u.startswith("http"):
            ws.cell(row=r, column=ti).hyperlink = u
            ws.cell(row=r, column=ti).font = LINK_FONT
    # Risk colour
    ri = COLS.index("Risk Rating")+1
    for r in range(2, ws.max_row+1):
        fill = RISK_FILLS.get(str(ws.cell(row=r, column=ri).value or "").upper())
        if fill: ws.cell(row=r, column=ri).fill = fill
    # Cluster colour
    ci = COLS.index("Cluster")+1
    for r in range(2, ws.max_row+1):
        fill = CLUSTER_FILLS.get(str(ws.cell(row=r, column=ci).value or ""))
        if fill: ws.cell(row=r, column=ci).fill = fill
    # Wrap wide cols
    for cn in WIDE_COLS:
        idx = COLS.index(cn)+1
        for r in range(2, ws.max_row+1):
            ws.cell(row=r, column=idx).alignment = WRAP_TOP
        ws.column_dimensions[get_column_letter(idx)].width = 55
    _auto_w(ws)
    for cn in WIDE_COLS:
        ws.column_dimensions[get_column_letter(COLS.index(cn)+1)].width = 55
    return len(recs)

# Master + per-cluster
n = _write_articles("Articles", articles)
print(f"  ✓ Articles: {n} rows × {len(COLS)} cols")
for cl in "ABCDE":
    sub = [a for a in articles if a["cluster"] == cl]
    n = _write_articles(f"Cluster {cl}", sub)
    print(f"  ✓ Cluster {cl}: {n} rows")

# Grounding Events
ws_ge = wb.create_sheet("Grounding Events")
ge_keys = []
seen = set()
for r in ge_records:
    for k in r:
        if k not in seen: ge_keys.append(k); seen.add(k)
ws_ge.append(ge_keys)
for rec in ge_records:
    ws_ge.append([rec.get(k, "") for k in ge_keys])
_style_hdr(ws_ge, len(ge_keys))
if "Source URL" in ge_keys:
    sui = ge_keys.index("Source URL")+1
    for r in range(2, ws_ge.max_row+1):
        u = str(ws_ge.cell(row=r, column=sui).value or "")
        if u.startswith("http"):
            ws_ge.cell(row=r, column=sui).hyperlink = u
            ws_ge.cell(row=r, column=sui).font = LINK_FONT
_auto_w(ws_ge)
print(f"  ✓ Grounding Events: {len(ge_records)} rows × {len(ge_keys)} cols")

# ── Pivot: Risk × Cluster ────────────────────────────────────────────────────
ws_r = wb.create_sheet("Risk Summary")
ratings = sorted({a["risk_rating"] for a in articles if a["risk_rating"]})
clusters = sorted({a["cluster"] for a in articles if a["cluster"]})
ws_r.append(["Cluster"] + ratings + ["Total"])
cnt = defaultdict(Counter)
for a in articles:
    if a["cluster"] and a["risk_rating"]: cnt[a["cluster"]][a["risk_rating"]] += 1
for cl in clusters:
    row = [cl] + [cnt[cl].get(r, 0) for r in ratings] + [sum(cnt[cl].values())]
    ws_r.append(row)
ws_r.append(["Total"] + [sum(cnt[c].get(r, 0) for c in clusters) for r in ratings]
             + [sum(sum(cnt[c].values()) for c in clusters)])
_style_hdr(ws_r, len(ratings)+2); _auto_w(ws_r)
print("  ✓ Risk Summary")

# ── Pivot: EO Summary ────────────────────────────────────────────────────────
ws_e = wb.create_sheet("EO Summary")
ws_e.append(["EO", "Articles", "Mean Salience", "HIGH+EXTREME Risk", "Supportive", "Unsupportive", "Neutral"])
eo_g = defaultdict(list)
for a in articles: eo_g[a["eo"] or "(unlinked)"].append(a)
for eo in sorted(eo_g, key=lambda x: (-len(eo_g[x]), x)):
    arts = eo_g[eo]
    sals = [a["salience"] for a in arts if a["salience"]]
    ws_e.append([
        eo, len(arts),
        round(sum(sals)/len(sals), 3) if sals else None,
        sum(1 for a in arts if a["risk_rating"] in ("HIGH", "EXTREME")),
        sum(1 for a in arts if _stance_bucket(a["stance"]) == "Supportive"),
        sum(1 for a in arts if _stance_bucket(a["stance"]) == "Unsupportive"),
        sum(1 for a in arts if _stance_bucket(a["stance"]) == "Neutral"),
    ])
_style_hdr(ws_e, 7); _auto_w(ws_e)
print("  ✓ EO Summary")

# ── Pivot: Publisher Summary ─────────────────────────────────────────────────
ws_p = wb.create_sheet("Publisher Summary")
ws_p.append(["Publisher", "Articles", "HIGH+EXTREME", "Supportive", "Unsupportive", "Mean Salience"])
pub_g = defaultdict(list)
for a in articles: pub_g[a["publisher"] or "(unknown)"].append(a)
for pub in sorted(pub_g, key=lambda x: (-len(pub_g[x]), x)):
    arts = pub_g[pub]
    sals = [a["salience"] for a in arts if a["salience"]]
    ws_p.append([
        pub, len(arts),
        sum(1 for a in arts if a["risk_rating"] in ("HIGH", "EXTREME")),
        sum(1 for a in arts if _stance_bucket(a["stance"]) == "Supportive"),
        sum(1 for a in arts if _stance_bucket(a["stance"]) == "Unsupportive"),
        round(sum(sals)/len(sals), 3) if sals else None,
    ])
_style_hdr(ws_p, 6); _auto_w(ws_p)
print("  ✓ Publisher Summary")

# ── Pivot: Stance × Cluster ─────────────────────────────────────────────────
ws_s = wb.create_sheet("Stance Summary")
buckets = ["Supportive", "Neutral", "Unsupportive", "Unknown"]
ws_s.append(["Cluster"] + buckets + ["Total"])
scnt = defaultdict(Counter)
for a in articles:
    if a["cluster"]: scnt[a["cluster"]][_stance_bucket(a["stance"])] += 1
for cl in clusters:
    ws_s.append([cl] + [scnt[cl].get(b, 0) for b in buckets] + [sum(scnt[cl].values())])
ws_s.append(["Total"] + [sum(scnt[c].get(b, 0) for c in clusters) for b in buckets]
            + [sum(sum(scnt[c].values()) for c in clusters)])
_style_hdr(ws_s, len(buckets)+2); _auto_w(ws_s)
print("  ✓ Stance Summary")

# ── Save ─────────────────────────────────────────────────────────────────────
wb.save(OUTPUT)
size_kb = OUTPUT.stat().st_size / 1024
print(f"\n✓ Saved: {OUTPUT} ({size_kb:.1f} KB)")
print(f"  Sheets: {', '.join(wb.sheetnames)}")
for s in wb.sheetnames:
    sheet = wb[s]
    print(f"    {s}: {sheet.max_row-1} rows × {sheet.max_column} cols")

✓ 293 articles loaded from bundle
  288/293 in clusters
  445 grounding events parsed
  ✓ Articles: 293 rows × 26 cols
  ✓ Cluster A: 61 rows
  ✓ Cluster B: 61 rows
  ✓ Cluster C: 74 rows
  ✓ Cluster D: 83 rows
  ✓ Cluster E: 9 rows
  ✓ Grounding Events: 445 rows × 10 cols
  ✓ Risk Summary
  ✓ EO Summary
  ✓ Publisher Summary
  ✓ Stance Summary

✓ Saved: /home/electric/workspace/RapporteurApp/data_validation_outputs/grounding_package.xlsx (518.6 KB)
  Sheets: Articles, Cluster A, Cluster B, Cluster C, Cluster D, Cluster E, Grounding Events, Risk Summary, EO Summary, Publisher Summary, Stance Summary
    Articles: 293 rows × 26 cols
    Cluster A: 61 rows × 26 cols
    Cluster B: 61 rows × 26 cols
    Cluster C: 74 rows × 26 cols
    Cluster D: 83 rows × 26 cols
    Cluster E: 9 rows × 26 cols
    Grounding Events: 445 rows × 10 cols
    Risk Summary: 6 rows × 6 cols
    EO Summary: 9 rows × 7 cols
    Publisher Summary: 66 rows × 6 cols
    Stance Summary: 6 rows × 6 cols


## 10. Kaliya Agenda Analysis by Subsection

Generate Kaliya agenda analysis reports broken down by subsection.

In [19]:
def run_clustering_with_images():
    """
    Execute clustering on media linkage results.
    Ensures Lamia image records are pulled for each article.
    """
    # Load media linkage cache
    media_linkage_cache = Path("/mnt/workspace_2/bboa/workflows/media_linking/media_linkage_cache.jsonl")
    
    if not media_linkage_cache.exists():
        print(f"✗ Media linkage cache not found: {media_linkage_cache}")
        return False
    
    # Load linked media records
    linked_records = []
    with open(media_linkage_cache, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                linked_records.append(json.loads(line))
    
    print(f"✓ Loaded {len(linked_records)} media linkage records")
    
    # Verify image records from BBOA cache
    urls_with_images = 0
    for rec in linked_records:
        url = rec.get("article_url", "")
        norm_url = normalize_url(url)
        if norm_url in bboa_cache_urls_post:
            urls_with_images += 1
    
    image_coverage = (urls_with_images / len(linked_records) * 100) if linked_records else 0
    print(f"✓ Image coverage: {image_coverage:.1f}% ({urls_with_images}/{len(linked_records)})")
    
    # TODO: Add clustering algorithm invocation here
    print("\n⚠ Clustering implementation: Add your clustering algorithm here")
    print("  - Use linked_records for article-to-EO mappings")
    print("  - Pull images from BBOA cache via normalized URLs")
    print("  - Cluster by EO, domain, temporal patterns, etc.")
    
    return True

# Uncomment to run after media linkage
# clustering_success = run_clustering_with_images()

## 9. Clustering Analysis with Image Extraction

Cluster the linked media articles, ensuring Lamia image records are included.

In [20]:
import subprocess

def run_media_linkage():
    """Execute media linkage pipeline via rapporteur CLI."""
    cmd = [
        "python", "-m", "src.components.publisher.rapporteur_cli",
        "docs/CultureTechnologyProcurementMedia/document.yaml",
        "--media-link", "-v"
    ]
    
    print("Running media linkage...")
    print(f"Command: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd="/home/electric/workspace/RapporteurApp")
    
    if result.returncode == 0:
        print("✓ Media linkage completed successfully")
        # Extract summary from output
        for line in result.stdout.split("\n"):
            if "Phase" in line or "Summary" in line or "records" in line.lower():
                print(f"  {line.strip()}")
    else:
        print(f"✗ Media linkage failed with exit code {result.returncode}")
        print(result.stderr[-500:])  # Last 500 chars of error
    
    return result.returncode == 0

# Uncomment to run after achieving 100% alignment
# media_linkage_success = run_media_linkage()

## 8. Run Media Linkage (Post-Victory)

**Prerequisites**: Victory condition met (100% alignment)

Execute media linkage pipeline to connect media articles to EOs using the fully-aligned dataset.

## 7. Post-Processing: Validate 100% Alignment

**Run this section AFTER:**
- Processing `missing_indices.jsonl` through research-indexing-consumers
- Processing `missing_bboa.jsonl` through BBOA consumers + Lamia
- Restarting RAG service with updated indices

## 6. Export Missing URLs for Processing

## 5. Check BBOA Cache Coverage

## 4. Compare URLs: Perceptually Adjusted vs Indices

## 3. Load Existing Index Files (Summaries + QA Pairs)

## 2. Load Perceptually Adjusted URLs (Source of Truth)